# Analisi Genetica Squali Lamniformi

Questo notebook processa file FASTA contenenti sequenze genetiche (COI e NAD2) di squali Lamniformi
e importa i dati in un database MySQL insieme ai dati fenotipici.

## Workflow:
1. Parsing file FASTA → DataFrame
2. Unione dati genetici COI e NAD2
3. Export in CSV
4. Import in database MySQL (tabelle: genetic_sequences, phenotypic_data)

In [ ]:
# Import librerie necessarie
from Bio import SeqIO
import pandas as pd
import re

## Configurazione Percorsi File

In [ ]:
# Percorsi file di input e output
coi_fasta_file = "ress/COI.fasta"
nad2_fasta_file = "ress/NAD2.fasta"
genetic_output_csv = "lamniformes_genetic_data.csv"

## Parsing File FASTA

Funzione per leggere file FASTA ed estrarre:
- Nome scientifico (Genere specie) dall'ID della sequenza
- Sequenza nucleotidica

In [ ]:
def parse_fasta_to_dataframe(fasta_file_path, sequence_col_name):
    """
    Parsa un file FASTA, estrae il nome scientifico completo (Genere specie) dall'ID,
    pulendo eventuali caratteri non standard o spazi extra nell'ID, e la sequenza.
    """
    data = []
    print(f"\n--- Tentativo di leggere il file FASTA: {fasta_file_path} ---")
    try:
        for record in SeqIO.parse(fasta_file_path, "fasta"):
            
            full_id_string = record.description.strip()
            cleaned_id = re.sub(r'\s+', ' ', full_id_string).strip()

            # Estrae nome scientifico (Genere specie)
            match = re.match(r'^([A-Z][a-z]+)\s+([a-z]+).*', cleaned_id)

            if match:
                specie = f"{match.group(1)} {match.group(2)}"
            else:
                # Fallback: prende le prime due parole
                parts = cleaned_id.split(' ')
                if len(parts) >= 2:
                    specie = f"{parts[0]} {parts[1]}"
                else:
                    specie = cleaned_id
                print(f"AVVISO: Formato ID non standard per: '{record.id}'. Estratto: '{specie}'.")

            sequenza = str(record.seq)
            data.append({"Nome_Scientifico": specie, sequence_col_name: sequenza})

        print(f"File {fasta_file_path} letto con successo. Trovate {len(data)} sequenze.")
        return pd.DataFrame(data)

    except FileNotFoundError:
        print(f"ERRORE GRAVE: Il file '{fasta_file_path}' non è stato trovato.")
        print(f"Assicurati che il nome del file sia corretto e che si trovi nel percorso specificato.")
        return None
    except Exception as e:
        print(f"ERRORE generico durante la lettura di {fasta_file_path}: {e}")
        return None


# Parsing file COI
coi_df = parse_fasta_to_dataframe(coi_fasta_file, "COI_Sequence")

if coi_df is not None:
    print("\n--- Visualizzazione del DataFrame 'coi_df' (prime 5 righe) ---")
    print(coi_df.head())
    print(f"Dimensioni di coi_df: {coi_df.shape}")
    print(f"Colonne di coi_df: {coi_df.columns.tolist()}")
    print("-" * 50)
else:
    print("DataFrame 'coi_df' non è stato creato a causa di errori.")

# Parsing file NAD2
nad2_df = parse_fasta_to_dataframe(nad2_fasta_file, "NAD2_Sequence")

if nad2_df is not None:
    print("\n--- Visualizzazione del DataFrame 'nad2_df' (prime 5 righe) ---")
    print(nad2_df.head())
    print(f"Dimensioni di nad2_df: {nad2_df.shape}")
    print(f"Colonne di nad2_df: {nad2_df.columns.tolist()}")
    print("-" * 50)
else:
    print("DataFrame 'nad2_df' non è stato creato a causa di errori.")


# Unione dei due DataFrame genetici
print("\n--- Tentativo di unione dei DataFrame genetici (COI e NAD2) ---")
if coi_df is not None and nad2_df is not None:
    genetic_df_combined = pd.merge(
        coi_df[["Nome_Scientifico", "COI_Sequence"]],
        nad2_df[["Nome_Scientifico", "NAD2_Sequence"]],
        on="Nome_Scientifico",
        how="outer"
    )

    print(f"DataFrame genetico combinato creato. Dimensioni: {genetic_df_combined.shape}")
    print("\n--- Visualizzazione del DataFrame 'genetic_df_combined' (prime 5 righe) ---")
    print(genetic_df_combined.head())
    print(f"Colonne di genetic_df_combined: {genetic_df_combined.columns.tolist()}")
    print("-" * 50)

    # Salvataggio in CSV
    genetic_df_combined.to_csv(genetic_output_csv, index=False)
    print(f"\nSalvataggio del dataset genetico combinato in: {genetic_output_csv}")

else:
    print("Impossibile creare il DataFrame genetico combinato a causa di errori nella lettura dei file FASTA.")

print("\nProcesso completato per i file FASTA.")

## Import Dati Genetici in MySQL

Caricamento del CSV generato nel database MySQL nella tabella `genetic_sequences`

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

genetic_output_csv = "lamniformes_genetic_data.csv"

# Configurazione connessione MySQL
MYSQL_USER = 'root'
MYSQL_PASSWORD = 'root'
MYSQL_HOST = 'localhost'
MYSQL_PORT = 3306
MYSQL_DATABASE = 'lamniformes_db'
table_name = "genetic_sequences"


print(f"Tentativo di caricare il file CSV: {genetic_output_csv}")
if not os.path.exists(genetic_output_csv):
    print(f"ERRORE: Il file '{genetic_output_csv}' non è stato trovato.")
    print("Assicurati che sia nella stessa directory del tuo notebook o che il percorso sia corretto.")
else:
    try:
        # Caricamento CSV
        genetic_df = pd.read_csv(genetic_output_csv)
        print(f"File '{genetic_output_csv}' caricato con successo. Dimensioni: {genetic_df.shape}")
        print("Prime 5 righe del DataFrame:\n", genetic_df.head())

        # Connessione a MySQL
        db_connection_str = f'mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
        engine = create_engine(db_connection_str)
        print(f"\nTentativo di connessione al database MySQL '{MYSQL_DATABASE}' su '{MYSQL_HOST}:{MYSQL_PORT}'...")

        # Creazione database se non esiste
        try:
            temp_engine = create_engine(f'mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/')
            with temp_engine.connect() as conn:
                conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}"))
                conn.commit()
            print(f"Database '{MYSQL_DATABASE}' assicurato/creato.")
        except Exception as e:
            print(f"AVVISO: Impossibile creare/assicurare il database '{MYSQL_DATABASE}'. Potrebbe esistere già o esserci un problema di permessi: {e}")
            print("Procedo con la connessione al database specificato, supponendo che esista.")

        # Import dati nella tabella
        genetic_df.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"DataFrame importato con successo nella tabella '{table_name}' nel database '{MYSQL_DATABASE}'.")

        # Verifica import
        print(f"\nVerifica: Lettura dei dati dalla tabella '{table_name}' da MySQL...")
        with engine.connect() as connection:
            result = connection.execute(text(f"SELECT * FROM {table_name} LIMIT 5"))
            verified_df = pd.DataFrame(result.fetchall(), columns=result.keys())
            print("Prime 5 righe lette dal database MySQL:\n", verified_df.head())
            print(f"Numero totale di righe nel database MySQL: {pd.read_sql(f'SELECT COUNT(*) FROM {table_name}', engine).iloc[0,0]}")

        print("\nOperazione completata: Dati importati nel database MySQL.")

    except Exception as e:
        print(f"Si è verificato un errore durante l'importazione nel database MySQL. Controlla le tue credenziali e che il server sia attivo: {e}")

## Import Dati Fenotipici in MySQL

Caricamento dei dati fenotipici (caratteristiche morfologiche, distribuzione, conservazione) nella tabella `phenotypic_data`

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

# Percorso file CSV fenotipico
phenotypic_csv_file = "ress/csv_shark.csv"

# Configurazione MySQL (stessi parametri)
MYSQL_USER = 'root'
MYSQL_PASSWORD = 'root'
MYSQL_HOST = 'localhost'
MYSQL_PORT = 3306
MYSQL_DATABASE = 'lamniformes_db'

# Nome tabella per dati fenotipici
pheno_table_name = "phenotypic_data"

print(f"Tentativo di caricare il file CSV fenotipico: {phenotypic_csv_file}")
if not os.path.exists(phenotypic_csv_file):
    print(f"ERRORE: Il file '{phenotypic_csv_file}' non è stato trovato.")
    print("Assicurati che sia nella stessa directory del tuo notebook o che il percorso sia corretto.")
else:
    try:
        # Caricamento CSV con separatore punto e virgola
        phenotypic_df = pd.read_csv(phenotypic_csv_file, sep=';')
        print(f"File '{phenotypic_csv_file}' caricato con successo. Dimensioni: {phenotypic_df.shape}")
        print("Prime 5 righe del DataFrame fenotipico:\n", phenotypic_df.head())

        # Connessione al database
        db_connection_str = f'mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
        engine = create_engine(db_connection_str)
        print(f"\nConnessione al database MySQL '{MYSQL_DATABASE}' stabilita per importazione fenotipica.")

        # Import dati nella tabella
        phenotypic_df.to_sql(pheno_table_name, engine, if_exists='replace', index=False)
        print(f"DataFrame fenotipico importato con successo nella tabella '{pheno_table_name}' nel database '{MYSQL_DATABASE}'.")

        # Verifica import
        print(f"\nVerifica: Lettura dei dati dalla nuova tabella '{pheno_table_name}'...")
        with engine.connect() as connection:
            result = connection.execute(text(f"SELECT * FROM {pheno_table_name} LIMIT 5"))
            verified_pheno_df = pd.DataFrame(result.fetchall(), columns=result.keys())
            print("Prime 5 righe lette dal database dalla tabella fenotipica:\n", verified_pheno_df.head())
            print(f"Numero totale di righe nella tabella '{pheno_table_name}': {pd.read_sql(f'SELECT COUNT(*) FROM {pheno_table_name}', engine).iloc[0,0]}")

        print("\nOperazione completata: Dati fenotipici importati nel database MySQL.")

    except Exception as e:
        print(f"Si è verificato un errore durante l'importazione dei dati fenotipici nel database MySQL. Controlla il file CSV e le credenziali: {e}")

## Conclusione

Il database MySQL `lamniformes_db` ora contiene:
- **Tabella `genetic_sequences`**: 17 specie con sequenze COI e NAD2
- **Tabella `phenotypic_data`**: Dati morfologici, distribuzione geografica, stato di conservazione IUCN

I dati sono pronti per essere analizzati e visualizzati con Power BI! 